# SmartRx Assist: Handwritten Prescription Recognition (CRNN + CTC)

**Goal.** Read handwritten medicine names from a photo so a patient who cannot read small print can hear them, and so the app can set up reminders.

**Pipeline implemented in this notebook**

```
photo -> Otsu binarise -> split into word images -> CRNN (CNN + BiLSTM) -> CTC decode -> text + confidence
                                                                                     |
                           RxNorm API (fuzzy drug-name correction) <-----------------+
                           OpenFDA API (safety warnings)           <-----------------+
```

**What each step proves** (the professor can follow the numbering):

| Step | What happens | Output you should see |
|---|---|---|
| 1 | Install libraries, check GPU | torch version, GPU yes/no |
| 2 | Write the shared pre-processing/decoding module | `ocr_core.py` |
| 3 | Settings | one config cell |
| 4 | Get the data **through the Kaggle API** (no manual download) | dataset path |
| 5 | Inspect + parse labels | folder tree, sample images |
| 6 | Train / validation / test split (by form, no leakage) | counts |
| 7 | Pre-processing demo | before / after images |
| 8 | Model | architecture + parameter count |
| 9 | Training | loss and error curves |
| 10 | **Testing**: accuracy, character error rate | numbers computed on unseen test data |
| 11 | **Prediction** on single words | predicted vs true + confidence |
| 12 | Full prescription page: segment, recognise, RxNorm + OpenFDA | annotated page, table |
| 13 | Export to ONNX | `crnn.onnx` + parity check |
| 14 | **Data storage in a DBMS (SQLite)** | tables + SQL queries |
| 15 | Serve the model as an **API** | live endpoint |
| 16 | **Test the API like Postman** + export a Postman collection | JSON responses |

> Every accuracy number is **computed here, at run time, on the held-out test split.** None is hard-coded.

## Step 1: Install and check the GPU
Runtime > Change runtime type > **T4 GPU** makes training much faster. It also runs on CPU, only slowly.

In [ ]:
!pip -q install kagglehub onnx onnxruntime fastapi "uvicorn[standard]" requests
import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())

## Step 2: Shared module `ocr_core.py`
Pre-processing, word segmentation, CTC decoding and the error metrics live in **one file**. The training code below and the production server (Step 15) import the same file, so what we train on is exactly what runs in the app.

In [ ]:
%%writefile ocr_core.py
"""
ocr_core.py: the pipeline shared by the training notebook and the live server.
Pure OpenCV / NumPy (+ onnxruntime for inference), so training and serving can never drift apart.

Pipeline:  page image -> Otsu binarise -> find word boxes -> crop -> resize to 32x128 -> CRNN -> CTC decode
"""
import json
import cv2
import numpy as np

IMG_H, IMG_W = 32, 128  # every word crop is resized to this
BLANK = 0               # CTC blank index; charset[i] is class i + 1


# ---------------------------------------------------------------- pre-processing
def to_gray(img: np.ndarray) -> np.ndarray:
    if img.ndim == 3:
        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY if img.shape[2] == 3 else cv2.COLOR_BGRA2GRAY)
    return img


def otsu(gray: np.ndarray) -> np.ndarray:
    """Otsu threshold. Returns ink = 0 (black) on paper = 255 (white), whatever the input polarity."""
    _, b = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if (b == 0).mean() > 0.5:  # more black than white means we picked up an inverted page
        b = 255 - b
    return b


def preprocess_word(img: np.ndarray, use_otsu: bool = True) -> np.ndarray:
    """One word image -> float32 array (1, 32, 128) in [0, 1], white background = 1."""
    g = to_gray(img)
    if use_otsu:
        g = otsu(g)
    h, w = g.shape[:2]
    new_w = int(min(IMG_W, max(1, round(w * IMG_H / max(h, 1)))))
    r = cv2.resize(g, (new_w, IMG_H), interpolation=cv2.INTER_AREA)
    canvas = np.full((IMG_H, IMG_W), 255, np.uint8)
    canvas[:, :new_w] = r
    return (canvas.astype(np.float32) / 255.0)[None]


# ---------------------------------------------------------------- word segmentation
def _runs(flags: np.ndarray, min_gap: int):
    """Runs of True in a 1-D bool array; runs separated by fewer than `min_gap` False values are merged."""
    runs, start, gap = [], None, 0
    for i, f in enumerate(flags):
        if f:
            if start is None:
                start = i
            gap = 0
            end = i
        elif start is not None:
            gap += 1
            if gap >= min_gap:
                runs.append((start, end + 1))
                start = None
    if start is not None:
        runs.append((start, end + 1))
    return runs


def _word_gap_threshold(strips, med: float) -> float:
    """
    Learns how wide a blank gap must be to separate words. Pools every gap on the page and splits them into
    'between letters' vs 'between words' (1-D two-means). Falls back to 0.45 x writing height when the gaps
    are not clearly bimodal (for example a page with a single word per line).
    """
    gaps = []
    for strip in strips:
        runs = _runs(strip.any(axis=0), 1)
        gaps += [runs[i + 1][0] - runs[i][1] for i in range(len(runs) - 1)]
    fallback = max(3.0, 0.45 * med)
    if len(gaps) < 4:
        return fallback
    g = np.sort(np.array(gaps, dtype=np.float64))
    best, thr = -1.0, fallback
    for cut in np.unique(g)[:-1]:
        lo, hi = g[g <= cut], g[g > cut]
        score = len(lo) * len(hi) * (hi.mean() - lo.mean()) ** 2  # between-class variance
        if score > best:
            best, thr = score, (lo.mean() + hi.mean()) / 2
    lo, hi = g[g <= thr], g[g > thr]
    return float(thr) if len(hi) and len(lo) and hi.mean() / max(lo.mean(), 1) >= 1.7 else fallback


def segment_lines(img: np.ndarray):
    """
    Projection-profile segmentation for a page of horizontal text:
    split lines at blank rows, then split words at wide blank column runs.
    Returns (binary_page, lines); each line is a list of (x, y, w, h) word boxes, left to right.
    Works on clean, roughly level text. Rotated or touching lines need a real text detector (CRAFT, Document AI).
    """
    b = otsu(to_gray(img))
    ink = b == 0
    n, _, stats, _ = cv2.connectedComponentsWithStats((255 - b), connectivity=8)
    heights = [stats[i, cv2.CC_STAT_HEIGHT] for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= 15]
    if not heights:
        return b, []
    med = float(np.median(heights))  # roughly the x-height of the writing
    spans = [(y0, y1) for y0, y1 in _runs(ink.any(axis=1), max(2, int(0.25 * med))) if ink[y0:y1].sum() >= 30]
    strips = [ink[y0:y1] for y0, y1 in spans]
    gap = _word_gap_threshold(strips, med)
    lines = []
    for (y0, _), strip in zip(spans, strips):
        boxes = []
        for x0, x1 in _runs(strip.any(axis=0), max(2, int(round(gap)))):
            rows = np.where(strip[:, x0:x1].any(axis=1))[0]
            h = rows[-1] - rows[0] + 1
            if (x1 - x0) * h >= max(60.0, med * med * 0.3):
                boxes.append((x0, y0 + int(rows[0]), x1 - x0, int(h)))
        if boxes:
            lines.append(boxes)
    return b, lines


def crop(binary: np.ndarray, box, pad: int = 3) -> np.ndarray:
    x, y, w, h = box
    H, W = binary.shape[:2]
    return binary[max(0, y - pad): min(H, y + h + pad), max(0, x - pad): min(W, x + w + pad)]


# ---------------------------------------------------------------- CTC decoding + metrics
def ctc_greedy(probs: np.ndarray, charset: list):
    """probs: (T, C) softmax output. Returns (text, confidence). Best-path decoding: collapse repeats, drop blanks."""
    best = probs.argmax(axis=1)
    out, prev, kept = [], BLANK, []
    for t, k in enumerate(best):
        if k != prev and k != BLANK:
            out.append(charset[k - 1])
        if k != BLANK:
            kept.append(probs[t, k])
        prev = k
    return "".join(out), float(np.mean(kept)) if kept else 0.0


def edit_distance(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def cer(preds, truths) -> float:
    """Character error rate = total edit distance / total reference characters."""
    return sum(edit_distance(p, t) for p, t in zip(preds, truths)) / max(1, sum(len(t) for t in truths))


def word_accuracy(preds, truths) -> float:
    return float(np.mean([p == t for p, t in zip(preds, truths)])) if truths else 0.0


# ---------------------------------------------------------------- ONNX inference
class OnnxRecognizer:
    def __init__(self, model_path: str, charset_path: str):
        import onnxruntime as ort
        self.sess = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
        self.charset = json.load(open(charset_path))

    def predict(self, word_images, use_otsu: bool = True):
        if not word_images:
            return []
        x = np.stack([preprocess_word(w, use_otsu) for w in word_images])
        probs = self.sess.run(None, {"image": x})[0]  # (B, T, C)
        return [ctc_greedy(p, self.charset) for p in probs]

    def read_page(self, img: np.ndarray):
        """Full page -> [{'text': 'Amoxicillin 500mg', 'confidence': 0.91}, ...] (the contract the phone app expects)."""
        binary, lines = segment_lines(img)
        result = []
        for boxes in lines:
            preds = self.predict([crop(binary, b) for b in boxes], use_otsu=False)  # already binary
            if preds:
                result.append({"text": " ".join(t for t, _ in preds), "confidence": float(np.mean([c for _, c in preds]))})
        return result

## Step 3: Settings
Change only this cell. For a quick demo use the defaults; for the best accuracy set `MAX_SAMPLES = None` (all words) and `EPOCHS = 40`.

In [ ]:
SEED = 42
USE_OTSU = True        # the phone app sends Otsu-binarised images, so we train on the same kind of image
MAX_SAMPLES = 40000    # None = use every word
EPOCHS = 20
BATCH = 64
LR = 1e-3
PATIENCE = 5           # stop early if validation error does not improve for this many epochs
MAX_LABEL_LEN = 20     # the model sees 32 time-steps, so very long words are dropped
USE_SYNTHETIC = False  # True = skip Kaggle and train on generated word images (demo only, not real handwriting)

import os, glob, json, time, random, sqlite3
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import ocr_core as oc

random.seed(SEED); np.random.seed(SEED)
ARTIFACTS = "artifacts"; os.makedirs(ARTIFACTS, exist_ok=True)
IMG_H, IMG_W = oc.IMG_H, oc.IMG_W

## Step 4: Get the dataset through the Kaggle **API** (no manual download or upload)
`kagglehub` downloads **directly into this Colab machine** when the cell runs. Nothing is stored in the repository, and nobody clicks "download" on kaggle.com.

It needs a free Kaggle API token once: Kaggle > Settings > *Create New Token*, then in Colab click the key icon and add two **Secrets**: `KAGGLE_USERNAME` and `KAGGLE_KEY`. The notebook reads them automatically.

Dataset: `nibinv23/iam-handwriting-word-database` (IAM handwriting words: single handwritten English words with their transcription).
If Kaggle is unreachable the notebook falls back to generated images so every later step can still be demonstrated. That fallback is clearly labelled and its accuracy is **not** a handwriting result.

In [ ]:
DATA_ROOT = None
if not USE_SYNTHETIC:
    try:
        try:
            from google.colab import userdata
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        except Exception:
            pass
        import kagglehub
        if not os.environ.get("KAGGLE_KEY"):
            kagglehub.login()   # prompts for username and key
        DATA_ROOT = kagglehub.dataset_download("nibinv23/iam-handwriting-word-database")
        print("Downloaded through the Kaggle API to:", DATA_ROOT)
    except Exception as e:
        print("Kaggle API not available:", repr(e))
        print("-> using SYNTHETIC data (demo only).")
        USE_SYNTHETIC = True

## Step 5: Look at the data and read the labels
The first cell prints the folder layout so the structure is visible. If this dataset's layout differs from the standard IAM one, only `load_iam()` below needs adjusting.

In [ ]:
def show_tree(root, depth=2, limit=8):
    root = str(root)
    for dp, dn, fn in os.walk(root):
        d = dp[len(root):].count(os.sep)
        if d > depth:
            dn[:] = []; continue
        print("  " * d + os.path.basename(dp) + "/", f"({len(fn)} files)" if fn else "")
        dn.sort()
        if len(dn) > limit:
            print("  " * (d + 1) + f"... {len(dn) - limit} more folders"); dn[:] = dn[:limit]

if not USE_SYNTHETIC:
    show_tree(DATA_ROOT)

In [ ]:
def load_iam(root):
    """Standard IAM layout: words.txt has one line per word image:
       <id> <ok|err> <graylevel> <components> x y w h <tag> <transcription>"""
    words_txt, images = None, {}
    for dp, _, fn in os.walk(root):
        for f in fn:
            if f == "words.txt":
                words_txt = os.path.join(dp, f)
            elif f.lower().endswith((".png", ".jpg", ".jpeg")):
                images[os.path.splitext(f)[0]] = os.path.join(dp, f)
    assert words_txt, "words.txt not found. Look at the tree above and adapt load_iam() to this dataset's label file."
    rows = []
    for line in open(words_txt, encoding="utf-8", errors="ignore"):
        if line.startswith("#") or not line.strip():
            continue
        p = line.strip().split(" ")
        if len(p) < 10 or p[1] != "ok":      # skip badly segmented words
            continue
        wid, text = p[0], " ".join(p[9:])
        if wid in images:
            rows.append(dict(path=images[wid], text=text, form="-".join(wid.split("-")[:2])))
    return pd.DataFrame(rows)

DRUGS = ["Amoxicillin","Paracetamol","Metformin","Atorvastatin","Amlodipine","Omeprazole","Ibuprofen","Azithromycin","Cetirizine",
         "Losartan","Pantoprazole","Salbutamol","Aspirin","Clopidogrel","Ciprofloxacin","Doxycycline","Prednisolone","Levothyroxine",
         "Montelukast","Diclofenac","Warfarin","Furosemide","Gabapentin"]
EXTRA = ["tablet","daily","after","food","twice","morning","night","500mg","250mg","650mg","10mg","1-0-1","0-1-0","days","with","water","take","once","before"]

def make_synthetic(n, out="synth"):
    """Fallback only: renders words in assorted fonts with rotation and noise. NOT real handwriting."""
    from PIL import Image, ImageDraw, ImageFont
    rng = random.Random(SEED); os.makedirs(out, exist_ok=True)
    bad = ("symbol", "emoji", "dingbat", "icon", "awesome")
    fonts = [f for f in sorted(glob.glob("/usr/share/fonts/truetype/**/*.ttf", recursive=True)) if not any(b in f.lower() for b in bad)]
    rows = []
    for i in range(n):
        text, size = rng.choice(DRUGS + EXTRA), rng.randint(28, 44)
        try:
            font = ImageFont.truetype(rng.choice(fonts), size) if fonts else ImageFont.load_default()
        except Exception:
            font = ImageFont.load_default()
        w = int(ImageDraw.Draw(Image.new("L", (1, 1))).textlength(text, font=font)) + 40
        im = Image.new("L", (w, size + 40), 255)
        ImageDraw.Draw(im).text((20, 14), text, font=font, fill=rng.randint(0, 70))
        a = np.array(im.rotate(rng.uniform(-4, 4), fillcolor=255)).astype(np.float32)
        a = np.clip(a + np.random.normal(0, 6, a.shape), 0, 255).astype(np.uint8)
        p = os.path.join(out, f"s{i:05d}.png"); cv2.imwrite(p, a)
        rows.append(dict(path=p, text=text, form=f"synth-{i // 25}"))
    return pd.DataFrame(rows)

df = make_synthetic(4000) if USE_SYNTHETIC else load_iam(DATA_ROOT)
print("DATASET:", "SYNTHETIC (demo only)" if USE_SYNTHETIC else "IAM words via Kaggle API", "| words:", len(df))
df.head()

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(12, 4))
for a, (_, r) in zip(ax.ravel(), df.sample(8, random_state=SEED).iterrows()):
    a.imshow(cv2.imread(r.path, cv2.IMREAD_GRAYSCALE), cmap="gray"); a.set_title(f'"{r.text}"'); a.axis("off")
plt.tight_layout(); plt.show()

## Step 6: Split into train / validation / test
Split **by form** (the page a word came from), not by word. If words from the same writer's page were in both training and test, the score would look better than it really is.
Train 80% (the model learns), validation 10% (choose when to stop), test 10% (touched **once**, at the end).

In [ ]:
df = df[df.text.str.len().between(1, MAX_LABEL_LEN)].reset_index(drop=True)
if MAX_SAMPLES and len(df) > MAX_SAMPLES:
    df = df.sample(MAX_SAMPLES, random_state=SEED).reset_index(drop=True)

forms = sorted(df.form.unique()); np.random.RandomState(SEED).shuffle(forms)
n = len(forms); train_f, val_f = set(forms[:int(.8 * n)]), set(forms[int(.8 * n):int(.9 * n)])
df["split"] = df.form.map(lambda f: "train" if f in train_f else "val" if f in val_f else "test")

charset = sorted(set("".join(df.text)))          # class i+1 <-> charset[i]; class 0 is the CTC blank
c2i = {c: i + 1 for i, c in enumerate(charset)}
print(df.split.value_counts().to_string()); print("characters:", len(charset), "->", "".join(charset))
df.text.str.len().hist(bins=20); plt.title("Word length distribution"); plt.xlabel("characters"); plt.show()

## Step 7: Pre-processing (what the model actually sees)
Grayscale, **Otsu threshold** (the same step the phone app runs), resize to 32 pixels high, pad to 128 wide. Because training uses the same function as the server, there is no train/serve mismatch.

In [ ]:
fig, ax = plt.subplots(2, 6, figsize=(14, 3.2))
for j, (_, r) in enumerate(df.sample(6, random_state=1).iterrows()):
    g = cv2.imread(r.path, cv2.IMREAD_GRAYSCALE)
    ax[0, j].imshow(g, cmap="gray"); ax[0, j].set_title(r.text); ax[0, j].axis("off")
    ax[1, j].imshow(oc.preprocess_word(g, USE_OTSU)[0], cmap="gray", vmin=0, vmax=1); ax[1, j].axis("off")
ax[0, 0].set_ylabel("original"); plt.suptitle("top: original   bottom: model input (32 x 128, Otsu)"); plt.tight_layout(); plt.show()

## Step 8: The model, a CRNN
* **CNN** (5 conv blocks) turns the 32x128 image into a sequence of 32 feature columns, left to right.
* **Bi-directional LSTM** (2 layers) reads that sequence forwards and backwards so each column knows its neighbours.
* **Linear + softmax** gives, for each of the 32 columns, a probability over *characters + blank*.
* **CTC loss** lets us train from just the word text, with no need to say which column holds which letter.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def augment(g):
    """Small random rotation / scale / shear so the model does not memorise exact pixels (training set only)."""
    h, w = g.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), np.random.uniform(-3, 3), np.random.uniform(0.9, 1.1))
    M[0, 1] += np.random.uniform(-0.15, 0.15)
    return cv2.warpAffine(g, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=255)

class WordDS(Dataset):
    def __init__(self, frame, train):
        self.p, self.t, self.train = frame.path.tolist(), frame.text.tolist(), train
    def __len__(self):
        return len(self.p)
    def __getitem__(self, i):
        g = cv2.imread(self.p[i], cv2.IMREAD_GRAYSCALE)
        if g is None:
            g = np.full((IMG_H, IMG_W), 255, np.uint8)
        if self.train:
            g = augment(g)
        x = torch.from_numpy(oc.preprocess_word(g, USE_OTSU))
        return x, torch.tensor([c2i[c] for c in self.t[i]], dtype=torch.long), self.t[i]

def collate(batch):
    xs, ys, ts = zip(*batch)
    return torch.stack(xs), torch.cat(ys), torch.tensor([len(y) for y in ys], dtype=torch.long), list(ts)

def loader(split, train):
    return DataLoader(WordDS(df[df.split == split], train), batch_size=BATCH, shuffle=train, collate_fn=collate,
                      num_workers=2, pin_memory=torch.cuda.is_available())
train_dl, val_dl, test_dl = loader("train", True), loader("val", False), loader("test", False)

class CRNN(nn.Module):
    def __init__(self, n_classes, hidden=256):
        super().__init__()
        def block(i, o):
            return [nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True)]
        self.cnn = nn.Sequential(
            *block(1, 64), nn.MaxPool2d(2, 2),                       # 64  x 16 x 64
            *block(64, 128), nn.MaxPool2d(2, 2),                     # 128 x  8 x 32
            *block(128, 256), *block(256, 256), nn.MaxPool2d((2, 1), (2, 1)),   # 256 x 4 x 32
            *block(256, 512), nn.MaxPool2d((2, 1), (2, 1)),          # 512 x  2 x 32
            nn.AdaptiveAvgPool2d((1, None)), nn.Dropout2d(0.1))      # 512 x  1 x 32
        self.rnn = nn.LSTM(512, hidden, num_layers=2, bidirectional=True, batch_first=True, dropout=0.25)
        self.fc = nn.Linear(hidden * 2, n_classes)
    def forward(self, x):                      # x: (B, 1, 32, 128)
        f = self.cnn(x).squeeze(2)             # (B, 512, 32)
        o, _ = self.rnn(f.permute(0, 2, 1))    # (B, 32, 2*hidden)
        return self.fc(o)                      # logits (B, 32, n_classes)

model = CRNN(len(charset) + 1).to(device)
print("parameters: %.2f M" % (sum(p.numel() for p in model.parameters()) / 1e6))
print("output shape for one batch (batch, time steps, classes):", tuple(model(torch.zeros(2, 1, IMG_H, IMG_W, device=device)).shape))

## Step 9: Training
Each epoch: forward pass, CTC loss, back-propagation (AdamW). After every epoch we measure the **validation** word accuracy and character error rate; the best model is saved and training stops early if it stops improving.

* **Word accuracy** = share of words predicted exactly right.
* **CER** (character error rate) = edits needed to fix the prediction / characters in the truth. Lower is better; 0.10 means 10 wrong characters per 100.

In [ ]:
ctc = nn.CTCLoss(blank=0, zero_infinity=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)

def run_ctc(logits, y, yl):
    lp = logits.float().log_softmax(2).permute(1, 0, 2)                    # (T, B, C)
    il = torch.full((lp.size(1),), lp.size(0), dtype=torch.long)
    return ctc(lp, y, il, yl)

@torch.no_grad()
def evaluate(dl):
    model.eval(); tot = n = 0; preds, truths, confs = [], [], []
    for x, y, yl, txt in dl:
        logits = model(x.to(device))
        tot += run_ctc(logits, y, yl).item() * x.size(0); n += x.size(0)
        for t, c in (oc.ctc_greedy(p, charset) for p in logits.softmax(2).cpu().numpy()):
            preds.append(t); confs.append(c)
        truths += txt
    return tot / n, oc.word_accuracy(preds, truths), oc.cer(preds, truths), preds, truths, confs

history, best, bad = [], 1e9, 0
for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); run = seen = 0
    for x, y, yl, _ in train_dl:
        loss = run_ctc(model(x.to(device)), y, yl)
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        run += loss.item() * x.size(0); seen += x.size(0)
    vloss, vacc, vcer, *_ = evaluate(val_dl); sched.step(vcer)
    history.append(dict(epoch=ep, train_loss=run / seen, val_loss=vloss, val_word_acc=vacc, val_cer=vcer))
    print(f"epoch {ep:2d} | train loss {run/seen:.3f} | val loss {vloss:.3f} | val word acc {vacc:.1%} | val CER {vcer:.3f} | {time.time()-t0:.0f}s")
    if vcer < best - 1e-4:
        best, bad = vcer, 0; torch.save(model.state_dict(), f"{ARTIFACTS}/crnn_best.pt")
    else:
        bad += 1
        if bad >= PATIENCE:
            print("early stopping: validation CER stopped improving"); break

In [ ]:
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
h.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax[0], title="CTC loss")
h.plot(x="epoch", y="val_cer", ax=ax[1], title="Validation character error rate (lower is better)")
h.plot(x="epoch", y="val_word_acc", ax=ax[2], title="Validation word accuracy")
plt.tight_layout(); plt.show()

## Step 10: Testing on data the model has never seen
Load the **best** checkpoint and score it on the held-out **test** split, once.

In [ ]:
model.load_state_dict(torch.load(f"{ARTIFACTS}/crnn_best.pt", map_location=device))
tloss, tacc, tcer, preds, truths, confs = evaluate(test_dl)
print(f"TEST  words: {len(truths)} | word accuracy: {tacc:.2%} | character error rate: {tcer:.3f} | CTC loss: {tloss:.3f}")

metrics = dict(dataset="SYNTHETIC (demo only)" if USE_SYNTHETIC else "IAM words (Kaggle API)",
               n_train=int((df.split == "train").sum()), n_val=int((df.split == "val").sum()), n_test=len(truths),
               epochs_run=len(history), best_val_cer=float(best), test_word_accuracy=float(tacc), test_cer=float(tcer))
json.dump(metrics, open(f"{ARTIFACTS}/metrics.json", "w"), indent=2)

res = pd.DataFrame(dict(truth=truths, pred=preds, conf=confs)); res["ok"] = res.truth == res.pred
print("\nAccuracy by word length"); print(res.groupby(pd.cut(res.truth.str.len(), [0, 3, 6, 9, 20])).ok.agg(["mean", "count"]).round(3).to_string())
print("\nMean confidence when right: %.2f | when wrong: %.2f" % (res[res.ok].conf.mean(), res[~res.ok].conf.mean()))
print("\nIf the app only accepts words above a confidence threshold:")
print(pd.DataFrame([dict(min_confidence=t, share_of_words_kept=(res.conf >= t).mean(),
                         accuracy_of_kept=res[res.conf >= t].ok.mean() if (res.conf >= t).any() else float("nan"))
                    for t in (0.0, 0.5, 0.7, 0.8, 0.9, 0.95)]).round(3).to_string(index=False))
res.sample(12, random_state=SEED).assign(result=lambda d: d.ok.map({True: "correct", False: "WRONG"}))

The **confidence table** is how the phone app decides to say *"I am not sure, check with your pharmacist."* Pick the threshold from this table instead of guessing.

## Step 11: Prediction on single words

In [ ]:
@torch.no_grad()
def predict_crops(crops, use_otsu=True):
    model.eval()
    x = torch.from_numpy(np.stack([oc.preprocess_word(c, use_otsu) for c in crops])).to(device)
    return [oc.ctc_greedy(p, charset) for p in model(x).softmax(2).cpu().numpy()]

sample = df[df.split == "test"].sample(8, random_state=3)
out = predict_crops([cv2.imread(p, cv2.IMREAD_GRAYSCALE) for p in sample.path])
fig, ax = plt.subplots(2, 4, figsize=(13, 4))
for a, (_, r), (t, c) in zip(ax.ravel(), sample.iterrows(), out):
    a.imshow(cv2.imread(r.path, cv2.IMREAD_GRAYSCALE), cmap="gray"); a.axis("off")
    a.set_title(f'truth "{r.text}"\npred  "{t}" ({c:.0%})', color="green" if t == r.text else "red", fontsize=10)
plt.tight_layout(); plt.show()

## Step 12: A whole prescription page, with the RxNorm and OpenFDA **APIs**
A real photo holds many words. We (1) find the words, (2) read each with the model, (3) correct likely drug names with the **NIH RxNorm API** (fuzzy match), and (4) fetch safety warnings from the **OpenFDA API**.

*Honest expectation:* the page below is typed text rendered in a font, and the model was trained on IAM handwriting (which is general English, not drug names). So expect mistakes on drug names, and that is exactly why the RxNorm step exists. Real prescription handwriting needs fine-tuning on prescription data.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
PAGE_LINES = ["Amoxicillin 500mg twice", "Paracetamol 650mg", "Metformin 500mg daily", "Atorvastatin 10mg night"]
font_path = next((f for f in ("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf") if os.path.exists(f)), None)
font = ImageFont.truetype(font_path, 40) if font_path else ImageFont.load_default()
pg = Image.new("L", (1100, 110 * len(PAGE_LINES) + 40), 255); d = ImageDraw.Draw(pg)
for i, t in enumerate(PAGE_LINES):
    d.text((40, 30 + i * 110), t, font=font, fill=25)
page = np.array(pg)
page_png = cv2.imencode(".png", page)[1].tobytes()            # reused when we test the API
binary, lines = oc.segment_lines(page)
print("lines found:", len(lines), "| words per line:", [len(l) for l in lines])
vis = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
for ln in lines:
    for (x, y, w, h) in ln:
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 160, 0), 2)
plt.figure(figsize=(10, 4.5)); plt.imshow(vis[..., ::-1]); plt.axis("off"); plt.title("Words found by the segmenter"); plt.show()

In [ ]:
import requests
def rxnorm_match(term):
    """NIH RxNorm fuzzy match. https://rxnav.nlm.nih.gov/REST/approximateTerm.json"""
    try:
        c = requests.get("https://rxnav.nlm.nih.gov/REST/approximateTerm.json", params={"term": term, "maxEntries": 1}, timeout=15).json() \
                    .get("approximateGroup", {}).get("candidate", [])
        if not c: return None
        name = requests.get(f"https://rxnav.nlm.nih.gov/REST/rxcui/{c[0]['rxcui']}/properties.json", timeout=15).json().get("properties", {}).get("name")
        return dict(rxcui=c[0]["rxcui"], name=name, score=float(c[0]["score"]))
    except Exception:
        return None

def openfda_warning(name):
    """OpenFDA drug label. https://api.fda.gov/drug/label.json"""
    try:
        r = requests.get("https://api.fda.gov/drug/label.json", params={"search": f'openfda.brand_name:"{name}" openfda.generic_name:"{name}"', "limit": 1}, timeout=15)
        if r.status_code != 200: return None
        lab = r.json()["results"][0]; txt = (lab.get("boxed_warning") or lab.get("warnings") or [""])[0]
        return txt[:160] or None
    except Exception:
        return None

recognized_rows = []
for ln in lines:
    crops = [oc.crop(binary, b) for b in ln]
    for (txt, conf) in predict_crops(crops, use_otsu=False):          # already binary
        row = dict(raw_ocr=txt, confidence=round(conf, 2), rxnorm_name=None, rxcui=None, rxnorm_score=None, fda_warning=None)
        if txt.isalpha() and len(txt) >= 4:                           # only try drug-like words
            m = rxnorm_match(txt)
            if m:
                row.update(rxnorm_name=m["name"], rxcui=m["rxcui"], rxnorm_score=m["score"], fda_warning=openfda_warning(m["name"].split()[0]))
        recognized_rows.append(row)
pd.DataFrame(recognized_rows)

## Step 13: Export to ONNX
ONNX is a portable model format. The server (and later a phone) runs it with `onnxruntime`, with no PyTorch needed. We check that the exported model gives the same answers as the original.

In [ ]:
class Export(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x).softmax(2)       # probabilities, shape (B, 32, classes)

cpu_model = model.to("cpu").eval(); dummy = torch.zeros(1, 1, IMG_H, IMG_W)
kw = dict(input_names=["image"], output_names=["probs"], dynamic_axes={"image": {0: "batch"}, "probs": {0: "batch"}}, opset_version=17)
try:
    torch.onnx.export(Export(cpu_model), dummy, f"{ARTIFACTS}/crnn.onnx", dynamo=False, **kw)
except TypeError:                                            # older torch has no `dynamo` argument
    torch.onnx.export(Export(cpu_model), dummy, f"{ARTIFACTS}/crnn.onnx", **kw)
json.dump(charset, open(f"{ARTIFACTS}/charset.json", "w"), ensure_ascii=False)

import onnxruntime as ort
sess = ort.InferenceSession(f"{ARTIFACTS}/crnn.onnx", providers=["CPUExecutionProvider"])
xb = torch.from_numpy(np.stack([oc.preprocess_word(cv2.imread(p, cv2.IMREAD_GRAYSCALE), USE_OTSU) for p in df[df.split == "test"].path[:16]]))
with torch.no_grad(): ref = Export(cpu_model)(xb).numpy()
diff = float(np.abs(sess.run(None, {"image": xb.numpy()})[0] - ref).max())
print(f"ONNX vs PyTorch, largest difference: {diff:.2e}  ->", "MATCH" if diff < 1e-3 else "CHECK EXPORT")
print("model file: %.1f MB" % (os.path.getsize(f"{ARTIFACTS}/crnn.onnx") / 1e6))
model.to(device);

## Step 14: Where the data is saved: a **DBMS** (SQLite here, PostgreSQL in production)
Three different things are often confused, so to be explicit:

| | What it is | Role in SmartRx |
|---|---|---|
| **On the phone** | AES-256 encrypted local store | The patient's own records live here first, so the app works offline |
| **DBMS (this step)** | A database server holding tables you query with SQL | The **backend** keeps the OCR results and the synced records so guardians and doctors can see them |
| **Postman** | A **tool for sending requests to an API** | It does **not store data**. It is only used to test the API (Step 16) |

SQLite is a real relational DBMS that needs no server install, which suits a demo. The SQL is the same idea in PostgreSQL.

In [ ]:
%%writefile schema.sql
CREATE TABLE IF NOT EXISTS scans (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  created_at TEXT NOT NULL,
  source TEXT,
  text TEXT NOT NULL,
  confidence REAL
);
CREATE TABLE IF NOT EXISTS records (
  id TEXT PRIMARY KEY,
  ts TEXT NOT NULL,
  month TEXT NOT NULL,          -- 'YYYY-MM', indexed for the month/year screens
  type TEXT NOT NULL,           -- started | dose_added | dose_taken | dose_missed | scan | sos
  dose_id TEXT,
  drug TEXT,
  detail TEXT
);
CREATE INDEX IF NOT EXISTS idx_records_month ON records(month);

In [ ]:
DB = "smartrx_demo.db"
if os.path.exists(DB): os.remove(DB)
con = sqlite3.connect(DB); con.executescript(open("schema.sql").read())

now = time.strftime("%Y-%m-%dT%H:%M:%S")
con.executemany("INSERT INTO scans(created_at, source, text, confidence) VALUES (?,?,?,?)",
                [(now, "notebook-demo", r["rxnorm_name"] or r["raw_ocr"], r["confidence"]) for r in recognized_rows])

# a few patient records spread over two months, to show the month / year grouping
demo = [("r1", "2026-09-28T08:01:00", "dose_taken", "Metformin"), ("r2", "2026-09-28T21:05:00", "dose_missed", "Atorvastatin"),
        ("r3", "2026-10-01T08:03:00", "dose_taken", "Metformin"), ("r4", "2026-10-01T14:10:00", "dose_taken", "Amlodipine"),
        ("r5", "2026-10-02T08:00:00", "dose_missed", "Metformin"), ("r6", "2026-10-03T09:00:00", "scan", None)]
con.executemany("INSERT OR IGNORE INTO records(id, ts, month, type, drug) VALUES (?,?,?,?,?)", [(i, t, t[:7], ty, dr) for i, t, ty, dr in demo])
con.commit()

print("Recent scans"); print(pd.read_sql("SELECT * FROM scans ORDER BY id DESC LIMIT 5", con).to_string(index=False))
print("\nRecords per month and type (what the Month and Year screen shows)")
print(pd.read_sql("SELECT month, type, COUNT(*) AS n FROM records GROUP BY month, type ORDER BY month, type", con).to_string(index=False))
print("\nAdherence per month")
print(pd.read_sql("""SELECT month,
        SUM(type='dose_taken') AS taken, SUM(type='dose_missed') AS missed,
        ROUND(100.0 * SUM(type='dose_taken') / NULLIF(SUM(type IN ('dose_taken','dose_missed')), 0)) AS adherence_pct
        FROM records GROUP BY month""", con).to_string(index=False))
con.close()

## Step 15: Serve the model as an API
The phone app and the guardian dashboard talk to one backend. The file below is the **same `app.py` that ships in the project's `backend/` folder**. It loads `crnn.onnx`, exposes the OCR endpoint the app already calls, and saves results and records into the database.

In [ ]:
%%writefile app.py
"""
SmartRx Assist API (FastAPI).
  * POST /v1/ocr/handwriting  : page image -> text lines, using the trained CRNN (ONNX)
  * POST /v1/records, GET /v1/records, GET /v1/records/summary : the patient's record log
  * GET  /v1/scans            : recent OCR results
Storage is SQLite (a real DBMS, zero setup). Point DB_PATH at a file, or swap sqlite3 for psycopg for PostgreSQL.
Run:  uvicorn app:app --host 0.0.0.0 --port 8000
Env:  MODEL_DIR (folder with crnn.onnx + charset.json), DB_PATH, API_TOKEN (optional bearer token)

Deliberately NOT implemented: /v1/alerts/sms, /v1/escalations. Those must call Twilio and run a timer worker;
a stub that returns "ok" without sending would make the phone think a guardian was alerted when none was.
"""
import base64
import os
import sqlite3
import time
from typing import List, Optional

import cv2
import numpy as np
from fastapi import Depends, FastAPI, Header, HTTPException, Query
from pydantic import BaseModel

from ocr_core import OnnxRecognizer

MODEL_DIR = os.getenv("MODEL_DIR", "artifacts")
DB_PATH = os.getenv("DB_PATH", "smartrx.db")
API_TOKEN = os.getenv("API_TOKEN", "")

SCHEMA = open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "schema.sql")).read()

app = FastAPI(title="SmartRx Assist API", version="1.0")
_recognizer: Optional[OnnxRecognizer] = None


def db() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


with db() as _c:
    _c.executescript(SCHEMA)


def auth(authorization: str = Header(default="")):
    if API_TOKEN and authorization != f"Bearer {API_TOKEN}":
        raise HTTPException(401, "Invalid or missing bearer token")


def recognizer() -> OnnxRecognizer:
    global _recognizer
    if _recognizer is None:
        model, charset = f"{MODEL_DIR}/crnn.onnx", f"{MODEL_DIR}/charset.json"
        if not (os.path.exists(model) and os.path.exists(charset)):
            raise HTTPException(503, f"Model files not found in '{MODEL_DIR}'. Train with the Colab notebook first.")
        _recognizer = OnnxRecognizer(model, charset)
    return _recognizer


# ------------------------------------------------------------------ models
class OcrRequest(BaseModel):
    format: str                      # 'gray8' (raw pre-processed pixels) or 'encoded' (PNG/JPEG bytes)
    data: str                        # base64
    width: Optional[int] = None      # gray8 only
    height: Optional[int] = None     # gray8 only
    mime: Optional[str] = None


class RecordIn(BaseModel):
    id: str
    ts: str
    type: str
    doseId: Optional[str] = None
    drug: Optional[str] = None
    detail: Optional[str] = None


class RecordsIn(BaseModel):
    records: List[RecordIn]


# ------------------------------------------------------------------ routes
@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": os.path.exists(f"{MODEL_DIR}/crnn.onnx"), "time": time.time()}


@app.post("/v1/ocr/handwriting", dependencies=[Depends(auth)])
def ocr(req: OcrRequest):
    raw = base64.b64decode(req.data)
    if req.format == "gray8":
        if not (req.width and req.height) or len(raw) != req.width * req.height:
            raise HTTPException(400, "gray8 needs width and height matching the byte count")
        img = np.frombuffer(raw, np.uint8).reshape(req.height, req.width)
    elif req.format == "encoded":
        img = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)
        if img is None:
            raise HTTPException(400, "Could not decode image (PNG / JPEG expected)")
    else:
        raise HTTPException(400, "format must be 'gray8' or 'encoded'")

    lines = recognizer().read_page(img)
    now = time.strftime("%Y-%m-%dT%H:%M:%S")
    with db() as c:
        c.executemany("INSERT INTO scans(created_at, source, text, confidence) VALUES (?,?,?,?)",
                      [(now, req.format, l["text"], l["confidence"]) for l in lines])
    return {"lines": lines}


@app.post("/v1/records", dependencies=[Depends(auth)])
def save_records(body: RecordsIn):
    """Idempotent: re-sending the same record id is ignored, so the phone can safely retry."""
    with db() as c:
        c.executemany(
            "INSERT OR IGNORE INTO records(id, ts, month, type, dose_id, drug, detail) VALUES (?,?,?,?,?,?,?)",
            [(r.id, r.ts, r.ts[:7], r.type, r.doseId, r.drug, r.detail) for r in body.records])
    return {"saved": len(body.records)}


@app.get("/v1/records", dependencies=[Depends(auth)])
def list_records(month: str = Query(..., pattern=r"^\d{4}-\d{2}$")):
    with db() as c:
        rows = c.execute("SELECT * FROM records WHERE month = ? ORDER BY ts", (month,)).fetchall()
    return {"month": month, "records": [dict(r) for r in rows]}


@app.get("/v1/records/summary", dependencies=[Depends(auth)])
def summary(month: str = Query(..., pattern=r"^\d{4}-\d{2}$")):
    with db() as c:
        counts = {r["type"]: r["n"] for r in c.execute(
            "SELECT type, COUNT(*) AS n FROM records WHERE month = ? GROUP BY type", (month,))}
    taken, missed = counts.get("dose_taken", 0), counts.get("dose_missed", 0)
    return {"month": month, "taken": taken, "missed": missed, "scans": counts.get("scan", 0), "sos": counts.get("sos", 0),
            "adherence_pct": round(100 * taken / (taken + missed)) if taken + missed else None}


@app.get("/v1/scans", dependencies=[Depends(auth)])
def recent_scans(limit: int = 20):
    with db() as c:
        rows = c.execute("SELECT * FROM scans ORDER BY id DESC LIMIT ?", (min(limit, 200),)).fetchall()
    return {"scans": [dict(r) for r in rows]}

In [ ]:
import threading, uvicorn
os.environ.update(MODEL_DIR=ARTIFACTS, DB_PATH="smartrx_api.db")
if os.path.exists("smartrx_api.db"): os.remove("smartrx_api.db")
server = uvicorn.Server(uvicorn.Config("app:app", host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start(); time.sleep(4)
BASE = "http://127.0.0.1:8000"
print(requests.get(f"{BASE}/health").json())

## Step 16: Test the API the way **Postman** does, then export a Postman collection
Postman is a desktop app for sending HTTP requests and reading the replies. The cell below sends the same requests from Python, so the professor sees the exact request and response. The next cell writes a ready-made **Postman collection** file to import.

In [ ]:
import base64
def show(title, r):
    print(f"--- {title}  ->  HTTP {r.status_code}"); print(json.dumps(r.json(), indent=2)[:900], "\n")

b64png = base64.b64encode(page_png).decode()
show("POST /v1/ocr/handwriting  (format: encoded, the original photo)",
     requests.post(f"{BASE}/v1/ocr/handwriting", json={"format": "encoded", "mime": "image/png", "data": b64png}))
show("POST /v1/ocr/handwriting  (format: gray8, what the phone sends after Otsu)",
     requests.post(f"{BASE}/v1/ocr/handwriting", json={"format": "gray8", "width": binary.shape[1], "height": binary.shape[0],
                                                       "data": base64.b64encode(binary.tobytes()).decode()}))
show("POST /v1/records  (phone syncs its record log)",
     requests.post(f"{BASE}/v1/records", json={"records": [
         {"id": "a1", "ts": "2026-10-01T08:00:00", "type": "dose_taken", "drug": "Metformin"},
         {"id": "a2", "ts": "2026-10-02T08:00:00", "type": "dose_missed", "drug": "Metformin"}]}))
show("GET /v1/records/summary?month=2026-10", requests.get(f"{BASE}/v1/records/summary", params={"month": "2026-10"}))
show("GET /v1/scans", requests.get(f"{BASE}/v1/scans", params={"limit": 3}))

In [ ]:
import base64
def req(name, method, path, body=None, query=None):
    url = {"raw": "{{baseUrl}}" + path, "host": ["{{baseUrl}}"], "path": path.strip("/").split("/")}
    if query: url["query"] = [{"key": k, "value": v} for k, v in query.items()]
    r = {"method": method, "header": [{"key": "Content-Type", "value": "application/json"}], "url": url}
    if body is not None: r["body"] = {"mode": "raw", "raw": json.dumps(body, indent=2)}
    return {"name": name, "request": r, "event": [{"listen": "test", "script": {"type": "text/javascript",
            "exec": ["pm.test('status is 200', () => pm.response.to.have.status(200));"]}}]}

b64 = base64.b64encode(page_png).decode()
collection = {
    "info": {"name": "SmartRx Assist API", "schema": "https://schema.getpostman.com/json/collection/v2.1.0/collection.json"},
    "auth": {"type": "bearer", "bearer": [{"key": "token", "value": "{{token}}", "type": "string"}]},
    "variable": [{"key": "baseUrl", "value": "http://localhost:8000"}, {"key": "token", "value": ""}],
    "item": [
        req("Health", "GET", "/health"),
        req("OCR: prescription photo", "POST", "/v1/ocr/handwriting", {"format": "encoded", "mime": "image/png", "data": b64}),
        req("Save records", "POST", "/v1/records", {"records": [{"id": "p1", "ts": "2026-10-01T08:00:00", "type": "dose_taken", "drug": "Metformin"}]}),
        req("Records for a month", "GET", "/v1/records", query={"month": "2026-10"}),
        req("Month summary", "GET", "/v1/records/summary", query={"month": "2026-10"}),
        req("Recent scans", "GET", "/v1/scans", query={"limit": "10"}),
    ]}
json.dump(collection, open(f"{ARTIFACTS}/smartrx.postman_collection.json", "w"), indent=2)
print("Saved", f"{ARTIFACTS}/smartrx.postman_collection.json", "(Postman > Import). Set baseUrl to your server; leave token empty unless API_TOKEN is set.")

**Optional: let a real Postman on your laptop reach this Colab server.** Colab is not on the public internet; a free Cloudflare quick tunnel (no account) gives a temporary URL. Remove the `#` to run it, then paste the printed `https://....trycloudflare.com` into Postman's `baseUrl`.

In [ ]:
# !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
# !./cloudflared tunnel --url http://127.0.0.1:8000

## Step 17: Results and package the files

In [ ]:
m = json.load(open(f"{ARTIFACTS}/metrics.json"))
print(pd.Series(m).to_string())
print("\nFiles to download from the 'artifacts' folder: crnn.onnx, charset.json, metrics.json, smartrx.postman_collection.json")
try:
    from google.colab import files
    !zip -qr smartrx_artifacts.zip artifacts && echo zipped
    files.download("smartrx_artifacts.zip")
except Exception:
    pass

## Limitations, stated plainly
1. **IAM is general English handwriting, not doctors' prescriptions.** Drug names are rare in it, and prescription handwriting is more rushed and abbreviated. Expect noticeably lower accuracy on real prescriptions until the model is **fine-tuned on prescription images** (this needs a labelled set, with patient privacy handled properly).
2. **Word segmentation is classical** (blank rows and gaps). It works on clean, level text. Skewed, crowded or photographed-at-an-angle pages need deskewing or a text detector (CRAFT, Google Document AI).
3. **Never act on OCR alone.** The app already requires a person to confirm each medicine, shows RxNorm/OpenFDA checks, and flags low confidence. Step 10's confidence table is how to choose that threshold.
4. **RxNorm and OpenFDA cover US drugs.** Many Indian brand names will not match; add a local brand-to-generic table.
5. Training is stochastic; numbers vary slightly between runs. The seed is fixed for repeatability on the same hardware.